IMPORTING ALL REQUIRED LIBRARIES

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GridSearchCV , RandomizedSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score,f1_score,precision_score,recall_score,classification_report,confusion_matrix,roc_auc_score

In [2]:
df = pd.read_csv("data/churn_data.csv")
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"] , errors = "coerce")
df["TotalCharges"] = df["TotalCharges"].fillna(df["TotalCharges"].mean() , inplace = True)


C:\Users\shaik\AppData\Local\Temp\ipykernel_1684\716199326.py:2: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df["TotalCharges"] = df["TotalCharges"].fillna(df["TotalCharges"].mean() , inplace = True)


In [4]:
df["TotalServices"] = (
    (df["PhoneService"] == "Yes").astype(int) +
    (df["OnlineSecurity"] == "Yes").astype(int) +
    (df["TechSupport"] == "Yes").astype(int) +
    (df["StreamingTV"] == "Yes").astype(int) +
    (df["StreamingMovies"] == "Yes").astype(int)
)

In [5]:
important_features = [
    "Contract",
    "InternetService",
    "tenure",
    "MonthlyCharges",
    "TechSupport",
    "PaymentMethod",
    "Dependents",
    "TotalServices",
    "TotalCharges"
]

x = df[important_features]
y = df["Churn"]
x.head()

,Contract,InternetService,tenure,MonthlyCharges,TechSupport,PaymentMethod,Dependents,TotalServices,TotalCharges
0,Month-to-month,DSL,1,29.85,No,Electronic check,No,0,29.85
1,One year,DSL,34,56.95,No,Mailed check,No,2,1889.50
2,Month-to-month,DSL,2,53.85,No,Mailed check,No,2,108.15
3,One year,DSL,45,42.30,Yes,Bank transfer (automatic),No,2,1840.75
4,Month-to-month,Fiber optic,2,70.70,No,Electronic check,No,1,151.65


In [6]:
y = y.map({"Yes" : 1 , "No" : 0})
y.head()

0    0
1    0
2    1
3    0
4    1
Name: Churn, dtype: int64

MAKING A PIPELINE

In [7]:
numerical_process = Pipeline(
    [
        ("imputer" , SimpleImputer(strategy = "mean")),
        ("scale" , StandardScaler())
    ]
)

categorical_process = Pipeline(
    [
        ("imputer_caterical" , SimpleImputer(strategy = "most_frequent")),
        ("onehot" , OneHotEncoder(handle_unknown = "ignore" , sparse_output = False))
    ]
)

In [8]:
num_features = ["tenure","MonthlyCharges","TotalCharges" , "TotalServices"]
cat_features = x.select_dtypes(include = "object").columns
cat_features

C:\Users\shaik\AppData\Local\Temp\ipykernel_1684\346168201.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_features = x.select_dtypes(include = "object").columns


Index(['Contract', 'InternetService', 'TechSupport', 'PaymentMethod',
       'Dependents'],
      dtype='str')

In [9]:
preprocessor = ColumnTransformer(
    [
        ("numerical_process" , numerical_process ,num_features),
        ("categorical_process" , categorical_process ,cat_features)
    ]
)

In [10]:
x_train , x_test , y_train , y_test = train_test_split(x , y , test_size = 0.2 , random_state = 42)
x_train.head()


,Contract,InternetService,tenure,MonthlyCharges,TechSupport,PaymentMethod,Dependents,TotalServices,TotalCharges
2142,One year,DSL,21,64.85,No,Mailed check,Yes,3,1336.800000
1623,Two year,Fiber optic,54,97.20,No,Bank transfer (automatic),No,3,5129.450000
6074,Month-to-month,DSL,1,23.45,No,Electronic check,No,0,23.450000
1362,Month-to-month,Fiber optic,4,70.20,No,Electronic check,No,1,237.950000
6754,Two year,DSL,0,61.90,Yes,Bank transfer (automatic),Yes,3,2283.300441


In [11]:
x_train_processed = preprocessor.fit_transform(x_train)
x_test_processed = preprocessor.transform(x_test)

features = preprocessor.get_feature_names_out()

x_train_processed = pd.DataFrame(x_train_processed , columns = features)
x_test_processed = pd.DataFrame(x_test_processed , columns = features)

x_train_processed.head()



,numerical_process__tenure,numerical_process__MonthlyCharges,numerical_process__TotalCharges,numerical_process__TotalServices,categorical_process__Contract_Month-to-month,categorical_process__Contract_One year,categorical_process__Contract_Two year,categorical_process__InternetService_DSL,categorical_process__InternetService_Fiber optic,categorical_process__InternetService_No,categorical_process__TechSupport_No,categorical_process__TechSupport_No internet service,categorical_process__TechSupport_Yes,categorical_process__PaymentMethod_Bank transfer (automatic),categorical_process__PaymentMethod_Credit card (automatic),categorical_process__PaymentMethod_Electronic check,categorical_process__PaymentMethod_Mailed check,categorical_process__Dependents_No,categorical_process__Dependents_Yes
0,-0.465683,-0.000474,-0.422099,0.568383,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
1,0.885537,1.074754,1.255366,0.568383,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
2,-1.284605,-1.376499,-1.002985,-1.717321,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
3,-1.161766,0.177346,-0.908113,-0.955419,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
4,-1.325551,-0.098524,-0.003468,0.568383,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0


In [12]:
from imblearn.over_sampling import SMOTE

sm = SMOTE(random_state = 42)

x_train_smote , y_train_smote = sm.fit_resample(x_train_processed , y_train)

print("before : " , y_train.value_counts())

print("after : " , y_train_smote.value_counts())

before :  Churn
0    4138
1    1496
Name: count, dtype: int64
after :  Churn
0    4138
1    4138
Name: count, dtype: int64


MODELS TRAINING

In [13]:
model_rfc = RandomForestClassifier(n_estimators = 600 ,criterion='gini', max_depth = 7 , min_samples_split = 10 , min_samples_leaf = 5 ,random_state = 42 , class_weight = "balanced")

model_rfc.fit(x_train_smote , y_train_smote)

y_pred = model_rfc.predict(x_test_processed)
accuracy = accuracy_score(y_test , y_pred)
f1 = f1_score(y_test , y_pred)
report = classification_report(y_test , y_pred)
confusion = confusion_matrix(y_test , y_pred)
roc = roc_auc_score(y_test , y_pred)

print("accuracy : ", accuracy)
print("f1 : ", f1)
print("report : ", report)
print("confusion : ", confusion)
print("roc_auc_score : ",roc)

accuracy :  0.7721788502484032
f1 :  0.6617492096944152
report :                precision    recall  f1-score   support

           0       0.93      0.75      0.83      1036
           1       0.55      0.84      0.66       373

    accuracy                           0.77      1409
   macro avg       0.74      0.79      0.74      1409
weighted avg       0.83      0.77      0.78      1409

confusion :  [[774 262]
 [ 59 314]]
roc_auc_score :  0.7944636517022576
